# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is specified via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display basic info
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below, we enumerate each record set defined in the schema, and for each record set, its available fields and columns, referencing entities strictly by their `@id`.

In [ ]:
# List record sets and their components by @id
recordsets = list(dataset.record_sets)
print("Record sets (@id):")
for rs in recordsets:
    print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")
    if 'fields' in rs:
        print("  Fields (@id):")
        for f in rs['fields']:
            print(f"    * {f['@id']} (name: {f.get('name', 'N/A')})")
    if 'columns' in rs:
        print("  Columns (@id):")
        for col in rs['columns']:
            print(f"    * {col['@id']} (name: {col.get('name', 'N/A')})")

# Print the first record from each record set, referencing its @id
for rs in recordsets:
    print(f"\nExample records for record set {rs['@id']}:")
    try:
        records = list(dataset.records(record_set=rs['@id']))
        for x in records[:1]:
            print(x)
    except Exception as e:
        print(f"Could not load records: {e}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

We will reference each record set by its `@id` and create a dictionary of DataFrames for easy access.

In [ ]:
# Extract all record sets into pandas DataFrames
dataframes = {}
record_set_ids = [rs['@id'] for rs in recordsets]

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Show columns of the first loaded record set
if record_set_ids:
    df_preview_id = record_set_ids[0]
    print(f"Columns in DataFrame for record set {df_preview_id} (@id):")
    print(dataframes[df_preview_id].columns.tolist())
    print("Preview:")
    display(dataframes[df_preview_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we filter, normalize, and group records by their attributes using their explicit `@id`s.

In [ ]:
# Identify a numeric field for analysis
# E.g., we search for a field with 'Age' or similar in the columns of the first record set
df_id = df_preview_id
numeric_field = None
for col in dataframes[df_id].columns:
    if 'age' in col.lower():
        numeric_field = col
        break

if numeric_field:
    print(f"Numeric field identified (@id): {numeric_field}")
    threshold = 50
    filtered_df = dataframes[df_id][dataframes[df_id][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field, e.g. 'sex' or 'anatomical location' if present
    grouping_field = None
    for col in dataframes[df_id].columns:
        if 'sex' in col.lower() or 'anatomical' in col.lower():
            grouping_field = col
            break

    if grouping_field:
        grouped_df = filtered_df.groupby(grouping_field).mean(numeric_only=True)
        print(f"Grouped data by {grouping_field}:")
        display(grouped_df.head())
else:
    print("No numeric field identified for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Example: Plot the age distribution and compare by sex or anatomical location using field `@id` names.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=dataframes[df_id], x=numeric_field, bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Categorical comparison
    if grouping_field:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=grouping_field, y=numeric_field, data=dataframes[df_id])
        plt.title(f"{numeric_field} distribution by {grouping_field} (@id)")
        plt.xlabel(grouping_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded dataset metadata and record sets, referencing all entities strictly by `@id`.
- Extracted and previewed tabular records, identifying numeric and categorical fields for EDA.
- Filtered and normalized numeric data, grouped records for insight by categorical variables (all via their `@id`).
- Visualized data distributions and relationships.

**This notebook provides a reproducible workflow for FAIR^2 clinicopathological CRC dataset exploration with Croissant-compliant metadata using `mlcroissant`.**